In [ ]:

from load_data import load_data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = load_data()

print("Dataset shape:", df.shape)
print("Target variable: total_price")
display(df.head())


## 1. Numerical Relationships

This section examines how numerical features relate to total flight price. The goal is to identify linear and nonlinear patterns that may help determine which model family is appropriate.

All findings in this notebook are based on synthetic flight fare data.

In [ ]:

# Numerical features to investigate
numerical_features = [
    "days_to_departure",
    "distance_miles",
    "departure_hour",
    "aircraft_capacity",
    "seats_remaining",
    "load_factor",
    "competitor_min_price"
]

# Create scatterplots for each feature
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, feature in enumerate(numerical_features):
    axes[i].scatter(
        df[feature],
        df["total_price"],
        alpha=0.15,
        s=8
    )
    axes[i].set_title(f"{feature} vs. total_price")
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel("Total Price ($)")

# Hide the unused subplot
axes[-1].axis("off")

plt.tight_layout()
plt.show()


### Numerical Relationship Findings

The scatterplots reveal a combination of linear and nonlinear relationships in the synthetic dataset.

- **Days to departure:** Prices appear higher and more variable closer to departure. The relationship is not strictly linear, suggesting that booking lead time may influence ticket prices nonlinearly.
- **Distance:** Longer flights generally have higher prices, indicating a positive relationship between flight distance and total price.
- **Departure hour:** No strong linear relationship is visible. Prices vary across departure times.
- **Aircraft capacity:** Values cluster around a small number of aircraft capacities, and price distributions overlap considerably.
- **Seats remaining:** Higher prices are more common when fewer seats remain, suggesting a negative, potentially nonlinear relationship.
- **Load factor:** Higher load factors are generally associated with higher ticket prices, although the relationship is not strictly linear.
- **Competitor minimum price:** A strong positive relationship is visible, suggesting that competitor pricing may be a powerful predictor if it is available at prediction time.

The dataset contains extreme positive and negative price outliers that may affect statistical estimates and model training. These observations should be reviewed as part of the team's data-quality and target analyses.

Overall, the mixture of nonlinear relationships and overlapping feature distributions suggests that a tree-based model may be worth considering. However, model selection should also account for feature interactions, data leakage, time patterns, interpretability, computational cost, and the stakeholder's needs.

## 2. Feature Interactions

This section investigates whether relationships between numerical features and total flight price differ across categories. These interactions help determine whether a model needs to capture relationships beyond individual features.

In [ ]:

# Compare days to departure across cabin classes
plt.figure(figsize=(12, 6))

for cabin in df["cabin"].dropna().unique():
    subset = df[df["cabin"] == cabin].copy()

    # Group booking lead time into 10-day intervals
    subset["days_group"] = (
        subset["days_to_departure"] // 10
    ) * 10

    avg_prices = subset.groupby(
        "days_group"
    )["total_price"].median()

    plt.plot(
        avg_prices.index,
        avg_prices.values,
        marker="o",
        label=str(cabin)
    )

plt.title("Days to Departure vs. Median Price by Cabin")
plt.xlabel("Days to Departure")
plt.ylabel("Median Total Price ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Days to Departure × Cabin Class Findings

The graph shows that median ticket prices generally increase as departure approaches across all four cabin classes.

Business/First experiences the largest absolute price increase, with median fares rising from approximately $600 when booking 100–109 days in advance to nearly $1,200 within 0–9 days of departure.

Basic Economy and Main Cabin have lower median prices and smaller absolute increases, while Premium Economy falls between Main Cabin and Business/First.

The different slopes suggest an interaction between cabin class and days to departure. Booking lead time does not have the same relationship with ticket price across all cabin classes.

This supports considering models capable of capturing nonlinear relationships and feature interactions, such as Random Forest or XGBoost. However, the final model choice should also consider data leakage, interpretability, computational cost, and stakeholder requirements.

These findings are based on synthetic data and should not be interpreted as actual American Airlines pricing behavior.

In [ ]:

# Compare distance and median price for nonstop vs connecting flights
interaction_df = df[
    ["distance_miles", "nonstop", "total_price"]
].dropna().copy()

# Group flight distances into 250-mile intervals
interaction_df["distance_group"] = (
    interaction_df["distance_miles"] // 250
) * 250

plt.figure(figsize=(12, 6))

for nonstop_value in [True, False]:
    subset = interaction_df[
        interaction_df["nonstop"] == nonstop_value
    ]

    median_prices = subset.groupby(
        "distance_group"
    )["total_price"].median()

    label = "Nonstop" if nonstop_value else "Connecting"

    plt.plot(
        median_prices.index,
        median_prices.values,
        marker="o",
        label=label
    )

plt.title("Distance vs. Median Price by Flight Type")
plt.xlabel("Flight Distance (Miles)")
plt.ylabel("Median Total Price ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Distance × Nonstop Findings

Both nonstop and connecting flights show a strong, approximately linear increase in median ticket price as flight distance increases.

Nonstop flights consistently have higher median prices than connecting flights at similar distances. The price gap also increases with distance, from approximately $32 at 500 miles to nearly $100 at 2,500 miles.

The different slopes suggest an interaction between distance and nonstop status. Flight distance has a stronger relationship with ticket price for nonstop flights than for connecting flights.

This interaction could be captured through an explicit interaction term in a linear model or learned automatically by a tree-based model.

These findings are based on synthetic data and do not necessarily reflect real American Airlines pricing behavior.

## 3. Numerical Feature Correlations

This section examines correlations between numerical features and total flight price. Strong correlations may indicate useful predictive relationships or redundant variables.

Correlation alone does not establish causation or determine whether a feature is safe to use for prediction.

In [ ]:

# Select numeric columns
numeric_df = df.select_dtypes(include="number")

# Calculate Pearson correlations
correlation_matrix = numeric_df.corr()

# Display correlation matrix as a heatmap
fig, ax = plt.subplots(figsize=(12, 9))

heatmap = ax.imshow(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.columns)))

ax.set_xticklabels(
    correlation_matrix.columns,
    rotation=90
)
ax.set_yticklabels(correlation_matrix.columns)

plt.colorbar(heatmap, ax=ax, label="Pearson Correlation")
plt.title("Numerical Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Display correlations with target
target_correlations = (
    correlation_matrix["total_price"]
    .sort_values(ascending=False)
)

print("Correlations with total_price:")
print(target_correlations)


### Correlation Findings

The correlation matrix reveals several important relationships between numerical features and total flight price.

- **Competitor minimum price (r = 0.9322):** Shows a very strong positive correlation with total price. However, its availability at prediction time must be verified before using it as a predictor.
- **Base fare (r = 0.9320) and taxes/fees (r = 0.9217):** Both have very strong positive correlations with total price. Since they are components of the target, including them as predictors would introduce target leakage.
- **Distance (r = 0.4799):** Shows a moderate positive correlation, supporting the observation that longer flights generally cost more.
- **Days to departure (r = -0.1876):** Shows a weak negative linear correlation. Earlier plots suggest that the relationship may be nonlinear.
- **Aircraft capacity (r = 0.2075):** Shows a weak positive correlation.
- **Load factor (r = 0.0841) and seats remaining (r = -0.0363):** Show very weak linear correlations, although nonlinear relationships may still exist.

The heatmap also indicates a strong negative relationship between seats remaining and load factor, suggesting that these features contain overlapping information.

Overall, the dataset contains a mixture of linear relationships, nonlinear patterns, and potentially redundant features. Correlation alone is insufficient for selecting predictors.

The final feature set should exclude target-derived variables and follow the team's leakage audit.

## 4. Categorical Cardinality

This section examines the number of unique values in categorical features. High-cardinality features may create challenges for one-hot encoding and influence the choice of model family.

The analysis also distinguishes potentially useful categories from identifiers that should not be used as predictors.

In [ ]:

# Categorical features to investigate
categorical_features = [
    "origin",
    "destination",
    "carrier",
    "cabin",
    "fare_booking_class",
    "day_of_week"
]

# Count unique values and missing values
cardinality_results = []

for feature in categorical_features:
    cardinality_results.append({
        "Feature": feature,
        "Unique Values": df[feature].nunique(),
        "Missing Values": df[feature].isna().sum()
    })

cardinality_df = pd.DataFrame(cardinality_results)

# Sort by number of unique values
cardinality_df = cardinality_df.sort_values(
    "Unique Values",
    ascending=False
)

display(cardinality_df)

# Visualize categorical cardinality
plt.figure(figsize=(10, 5))

plt.bar(
    cardinality_df["Feature"],
    cardinality_df["Unique Values"]
)

plt.title("Categorical Feature Cardinality")
plt.xlabel("Categorical Feature")
plt.ylabel("Number of Unique Values")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Categorical Cardinality Findings

The categorical features have relatively low to moderate cardinality in the synthetic dataset.

- **Origin and destination:** Each contains 25 unique airports. One-hot encoding is feasible, although route-level interactions could increase feature complexity.
- **Fare booking class:** Contains 10 unique categories, making categorical encoding manageable.
- **Day of week:** Contains 7 categories, which may help capture weekly pricing patterns.
- **Cabin:** Contains 4 categories and is likely useful given the price differences observed across cabin classes.
- **Carrier:** Contains only 1 unique value, meaning it provides no variation and is a candidate for removal.

None of the six categorical columns contains missing values in this dataset.

Overall, categorical cardinality does not present a major obstacle to using regularized linear models or tree-based models. A simple one-hot encoding approach would be reasonable for the current categories.

Final feature selection should follow the team's leakage audit, and these findings may not generalize to the real American Airlines dataset.

## 5. Time Trends and Seasonality

This section investigates the dataset's time structure and potential temporal patterns.

The analysis examines date coverage, monthly price trends, and weekly seasonality to determine whether the dataset is better suited to tabular regression with time-related features or a traditional time-series forecasting model.

In [ ]:

# Check date coverage
print("Search date range:")
print(df["search_date"].min(), "to", df["search_date"].max())

print("\nDeparture date range:")
print(df["departure_date"].min(), "to", df["departure_date"].max())

print("\nUnique search dates:", df["search_date"].nunique())
print("Unique departure dates:", df["departure_date"].nunique())

# Monthly median prices by departure date
time_df = df[
    ["departure_date", "total_price"]
].dropna().copy()

time_df["departure_month"] = (
    time_df["departure_date"].dt.to_period("M")
)

monthly_prices = time_df.groupby(
    "departure_month"
)["total_price"].median()

# Plot monthly median prices
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_prices.index.astype(str),
    monthly_prices.values,
    marker="o"
)

plt.title("Monthly Median Flight Price by Departure Date")
plt.xlabel("Departure Month")
plt.ylabel("Median Total Price ($)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Time Trend and Seasonality Findings

The synthetic dataset contains 300 unique search dates from January 1 through October 27, 2026, and 417 unique departure dates from January 2, 2026, through February 23, 2027.

Monthly median ticket prices fluctuate throughout the observed period. January 2026 has the highest median price at approximately $330, followed by a decline to around $229 in April. Prices rise again near July before decreasing toward the end of the observed departure-date range.

These fluctuations suggest that calendar timing may be associated with flight prices. However, the dataset does not cover enough repeated annual cycles to establish reliable yearly seasonality.

Additionally, search dates end in October 2026, meaning later departures are generally observed further in advance. Since booking lead time is associated with ticket price, the apparent downward movement in later months may partly reflect differences in days to departure rather than a true temporal trend.

Overall, the dataset contains usable date-related features, but the current evidence does not establish that a traditional univariate time-series model would be more appropriate than tabular regression.

These findings are based on synthetic data and should not be interpreted as actual American Airlines pricing trends.

## 6. Day-of-Week Effects

This section examines whether median flight prices differ across departure weekdays.

Weekly pricing patterns may provide useful predictive information even if long-term annual seasonality cannot be established.

In [ ]:

# Define chronological weekday order
weekday_order = [
    "Mon", "Tue", "Wed", "Thu",
    "Fri", "Sat", "Sun"
]

# Calculate median price for each weekday
weekday_prices = (
    df.groupby(
        "day_of_week",
        observed=True
    )["total_price"]
    .median()
    .reindex(weekday_order)
)

# Display weekday median prices
print("Median ticket price by departure weekday:")
print(weekday_prices)

# Create bar chart
plt.figure(figsize=(10, 5))

plt.bar(
    weekday_prices.index,
    weekday_prices.values
)

plt.title("Median Flight Price by Departure Weekday")
plt.xlabel("Departure Weekday")
plt.ylabel("Median Total Price ($)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Day-of-Week Findings

Median ticket prices differ across departure weekdays in the synthetic dataset.

Monday through Thursday have relatively similar median prices, ranging from approximately $226.61 to $229.50. Friday, Saturday, and Sunday have higher median prices, ranging from approximately $247.10 to $249.60.

Tuesday has the lowest median price at $226.61, while Friday has the highest at $249.60, a difference of approximately $23.

These results suggest that departure weekday may provide useful predictive information. However, the differences are descriptive and may partly reflect other factors, including routes, cabin classes, flight distances, and booking lead times.

Day of week is therefore a reasonable candidate feature, subject to the team's leakage audit and final feature-selection process.

These findings are based on synthetic data and should not be interpreted as actual American Airlines pricing patterns.

## 7. Model Family Recommendation

### Recommended Model: XGBoost Regression

Based on the exploratory analysis of the synthetic dataset, XGBoost is the provisional recommended model family for predicting total flight price.

**1. Data Shape and Relationships**

The dataset contains 25,050 observations and 24 columns, including numerical, categorical, boolean, and date-related variables.

Several features exhibit nonlinear relationships with total price. The interaction analyses also suggest that booking lead time behaves differently across cabin classes and that the distance-price relationship depends on nonstop status.

Gradient-boosted trees can represent these nonlinear patterns and interactions without requiring every relationship to be manually specified.

**2. Loss Function**

Huber loss is a reasonable candidate because the dataset contains extreme positive and negative price outliers. It combines sensitivity to smaller errors with reduced sensitivity to large errors compared with squared-error loss.

However, the final loss function should follow the target/outlier analysis and the stakeholder's preferred error metric. Standard squared-error XGBoost remains an appropriate baseline.

**3. Stakeholder Needs**

American Airlines stakeholders would benefit from accurate predictions and understandable explanations of which features influence the forecasts.

XGBoost can support feature-importance analysis and SHAP-based interpretation. These tools can help communicate model behavior, although they do not establish causal effects.

**4. Computational and Maintenance Cost**

For a dataset containing approximately 25,000 observations, XGBoost should be computationally feasible.

It requires more tuning and maintenance than regularized linear regression, so its performance improvements must justify the additional complexity.

**5. Model Performance**

Model performance has not yet been established for the final leakage-safe feature set.

XGBoost should be evaluated against a simple baseline, Ridge regression, and Random Forest using the team's chosen validation strategy. A chronological split is preferable when the intended deployment involves predicting future observations.

The unusually high R² reported in Deliverable 1 should not be treated as evidence that XGBoost will generalize to the current dataset.

### Final Recommendation

XGBoost is recommended as the primary model family to investigate because it can capture nonlinear relationships and interactions in the tabular flight-fare dataset.

This recommendation is provisional until the leakage audit, target/loss analysis, and validation results are finalized.

All findings are based on synthetic data and should not be generalized to real American Airlines operations without additional analysis.